# Assignment Module 2 — Part 1: design your own network

| Student ID | Name | Email |
|---|---|---|
| ... | ... | ... |
| ... | ... | ... |

Image Processing and Computer Vision — Module 2

Goal: classify the 100 aircraft variants of **FGVC-Aircraft** with a convolutional network trained **from
scratch**, built only from PyTorch primitives (`nn.Conv2d`, `nn.BatchNorm2d`, `nn.Linear`, ...).

This version trains the same network with a **full modern training recipe**. Every ingredient addresses a problem
we measured on this dataset, and each is a standard published technique applied to the *training procedure* — no
pre-trained weights are used anywhere.

| ingredient | problem it solves | reference |
|---|---|---|
| SGD + Nesterov momentum, cosine schedule with warmup | with a constant learning rate the run never settles: the best epoch was 13.7 points above the final one | Loshchilov & Hutter, ICLR 2017 |
| RandomResizedCrop + horizontal flip | training accuracy reaches 100% by epoch ~70; random crops stop the network memorizing exact images | Simonyan & Zisserman, ICLR 2015 |
| TrivialAugmentWide | stronger, parameter-free augmentation | Müller & Hutter, ICCV 2021 |
| RandomErasing | occlusion robustness, "dropout of the input space" | Zhong et al., AAAI 2020 |
| Mixup / CutMix | force linear behaviour between classes; the strongest regularizers for small datasets | Zhang et al., ICLR 2018; Yun et al., ICCV 2019 |
| label smoothing | prevents over-confident logits | Szegedy et al., CVPR 2016 |
| weight decay, not applied to BatchNorm parameters and biases | limits the effective capacity | He et al., *Bag of Tricks*, CVPR 2019 |
| dropout before the classifier | same | Srivastava et al., JMLR 2014 |
| EMA of the weights | makes the final model the best model | Polyak & Juditsky, 1992 |
| mixed precision | roughly halves the epoch time | Micikevicius et al., ICLR 2018 |

## The dataset

FGVC-Aircraft labels every image at three levels of a hierarchy: **manufacturer** (e.g. Boeing), **family**
(e.g. Boeing 737) and **variant** (e.g. Boeing 737-700). We use `annotation_level="variant"`, the finest level of
the benchmark, which `torchvision` exposes as **100 classes** (the dataset page counts 102 variants in total).

A variant deliberately *collapses all the models that are visually indistinguishable into one class*, so the task
is well posed: every class can in principle be told apart from the image alone. The difficulty is therefore not
label ambiguity but

* **very little data**: 3,334 training / 3,333 validation / 3,333 test images, i.e. only **~33 training images per
  class**, so the network memorizes the training set long before it runs out of things to learn;
* **subtle, localized evidence**: variants of the same family differ by engine type, winglets, window or door
  layout — details that survive a resize to 320px far better than to 224px.

Every image also carries a ~20 pixel copyright banner at the bottom ("Photo by ... / Airliners.net"). We crop it
away, otherwise the network could pick up photographer-specific cues instead of aircraft shape. The crop is
deterministic and applied to all three splits; normalization statistics are computed on the **training split
only**, so no information leaks from validation or test.

## Import dependencies

In [ ]:
import copy
import math
import random
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sn

import torch
import torch.nn as nn
import torch.nn.functional as F

from tqdm.notebook import tqdm

from torch.utils.data import DataLoader
from torchvision import transforms as T, datasets

## Reproducibility and runtime

As in the lab sessions, we disable every source of randomness we can control, so that the comparison between
different configurations is not polluted by run-to-run variance.

In [ ]:
def fix_random(seed: int) -> None:
    """Fix all the possible sources of randomness.

    Args:
        seed: the seed to use.
    """
    np.random.seed(seed)
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)

    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

fix_random(42)

device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
print(f"Device: {device}")

## Data

Training images are randomly cropped and rescaled, flipped, augmented with TrivialAugmentWide, and finally a
random rectangle is erased. Validation and test images are only resized and normalized: their transform is
deterministic, identical for every image, and uses statistics computed on the training split.

In [ ]:
img_size = 320      # fine-grained details survive this resize much better than 224
batch_size = 64
epochs = 150
n_classes = 100

# channel statistics computed on the training split only
mean = (0.488, 0.519, 0.544)
std = (0.211, 0.203, 0.236)


class RemoveBanner:
    """Crop the ~20 pixel copyright banner at the bottom of every FGVC-Aircraft image."""

    def __call__(self, img):
        return img.crop((0, 0, img.width, img.height - 20))

In [ ]:
tsfms_train = T.Compose([
    RemoveBanner(),
    T.Resize((img_size, img_size)),
    T.RandomResizedCrop(img_size, scale=(0.35, 1.0), antialias=True),
    T.RandomHorizontalFlip(),
    T.TrivialAugmentWide(),
    T.ToTensor(),
    T.Normalize(mean, std),
    T.RandomErasing(p=0.25)
])

tsfms_eval = T.Compose([
    RemoveBanner(),
    T.Resize((img_size, img_size)),
    T.ToTensor(),
    T.Normalize(mean, std)
])

In [ ]:
train_dset = datasets.FGVCAircraft(
    root="./data",
    split="train",
    annotation_level="variant",
    transform=tsfms_train,
    download=True
)
val_dset = datasets.FGVCAircraft(
    root="./data",
    split="val",
    annotation_level="variant",
    transform=tsfms_eval,
    download=True
)
test_dset = datasets.FGVCAircraft(
    root="./data",
    split="test",
    annotation_level="variant",
    transform=tsfms_eval,
    download=True
)

train_dl = DataLoader(
    train_dset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=8,
    pin_memory=True,
    drop_last=True      # avoids a tiny last batch, which would make BatchNorm statistics noisy
)
val_dl = DataLoader(val_dset, batch_size=128, num_workers=8)
test_dl = DataLoader(test_dset, batch_size=128, num_workers=8)

print(f"Classes: {len(train_dset.classes)}")
print(f"Train: {len(train_dset)} - Validation: {len(val_dset)} - Test: {len(test_dset)}")

A batch of training images, after augmentation — note the crops, the colour changes and the erased patches:

In [ ]:
imgs, labels = next(iter(train_dl))

_, ax = plt.subplots(2, 5, figsize=(16, 7))

for i, a in enumerate(ax.flat):
    img = imgs[i].permute(1, 2, 0) * torch.tensor(std) + torch.tensor(mean)
    a.imshow(img.clamp(0, 1))
    a.set_title(train_dset.classes[labels[i]], fontsize=9)
    a.axis("off")

## The network

We arrived at this design by starting from a plain stack of convolutions and fixing one measured problem at a time:

1. **A plain CNN memorizes immediately** (training accuracy ~100%, validation ~21%), so generalization — not
   capacity — is the hard part.
2. **Global average pooling instead of flatten + fully connected.** In the plain CNN, 81% of all parameters sat in
   the first linear layer, which is where memorization happens. Averaging the spatial dimensions away removes them
   (Lin et al., *Network in Network*, ICLR 2014; used by GoogLeNet and ResNet).
3. **Batch Normalization after every convolution** (Ioffe & Szegedy, ICML 2015): without it the deeper network did
   not train at all in our experiments.
4. **Multi-branch blocks.** The evidence that separates two variants lives at several scales at once — overall
   fuselage proportions, but also engine, winglet and window detail. A block that applies 1x1, 3x3 and (factorized)
   5x5 convolutions *in parallel* and concatenates the results lets every layer pick its own scale, instead of
   committing to a single receptive field. The leading 1x1 convolutions reduce the channel count first, which keeps
   the block cheap (Szegedy et al., *Going Deeper with Convolutions*, CVPR 2015).
5. **A stem that downsamples early**, so the expensive blocks run on smaller feature maps.

The resulting network has four stages; each stage halves the spatial resolution and doubles the width:

```
stem        3x3 s2 -> 3x3 -> 3x3 -> maxpool          320 -> 80
stage 1     2 inception blocks, 160 channels          80
stage 2     2 inception blocks, 320 channels          40
stage 3     2 inception blocks, 640 channels          20
stage 4     2 inception blocks, 1280 channels         10
head        global average pool -> dropout -> linear(1280, 100)
```

In [ ]:
def get_conv_bn(in_channels: int, out_channels: int, kernel_size: int, stride: int = 1) -> nn.Sequential:
    """Return a Sequential block with a convolution followed by Batch Normalization.

    The convolution has no bias, since the Batch Normalization that follows it would cancel it out.

    Args:
        in_channels: number of input channels.
        out_channels: number of output channels.
        kernel_size: size of the (square) kernel.
        stride: stride of the convolution.

    Returns: the created Sequential block.
    """
    return nn.Sequential(
        nn.Conv2d(in_channels, out_channels, kernel_size, stride=stride,
                  padding=kernel_size // 2, bias=False),
        nn.BatchNorm2d(out_channels)
    )


class InceptionBlock(nn.Module):
    """Four parallel branches, concatenated along the channel dimension:

        1x1                     pointwise
        1x1 -> 3x3              medium receptive field
        1x1 -> 3x3 -> 3x3       5x5 receptive field, factorized into two 3x3
        maxpool -> 1x1          pooled view of the input

    The leading 1x1 convolutions reduce the number of channels before the expensive 3x3 ones.
    """

    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        quarter = out_channels // 4
        reduced = max(out_channels // 8, 8)

        self.branch_1x1 = get_conv_bn(in_channels, quarter, 1, stride)

        self.branch_3x3 = nn.Sequential(
            get_conv_bn(in_channels, reduced, 1), nn.ReLU(),
            get_conv_bn(reduced, quarter, 3, stride)
        )

        self.branch_5x5 = nn.Sequential(
            get_conv_bn(in_channels, reduced, 1), nn.ReLU(),
            get_conv_bn(reduced, quarter, 3), nn.ReLU(),
            get_conv_bn(quarter, quarter, 3, stride)
        )

        self.branch_pool = nn.Sequential(
            nn.MaxPool2d(3, stride=stride, padding=1),
            get_conv_bn(in_channels, out_channels - 3 * quarter, 1)
        )

    def forward(self, x):
        branches = [self.branch_1x1(x), self.branch_3x3(x), self.branch_5x5(x), self.branch_pool(x)]
        return F.relu(torch.cat(branches, dim=1))


class AircraftNet(nn.Module):
    """Stem, four stages of Inception blocks, global average pooling, linear classifier."""

    def __init__(self, n_classes, channels=(160, 320, 640, 1280), blocks_per_stage=2, dropout=0.3):
        super().__init__()
        first = channels[0]

        # the stem reduces 320x320 to 80x80 before the expensive blocks
        self.stem = nn.Sequential(
            get_conv_bn(3, first // 2, 3, stride=2), nn.ReLU(),
            get_conv_bn(first // 2, first // 2, 3), nn.ReLU(),
            get_conv_bn(first // 2, first, 3), nn.ReLU(),
            nn.MaxPool2d(3, stride=2, padding=1)
        )

        blocks = []
        in_channels = first
        for stage, out_channels in enumerate(channels):
            for block in range(blocks_per_stage):
                # the first block of every stage (except the first one) halves the resolution
                stride = 2 if (block == 0 and stage > 0) else 1
                blocks.append(InceptionBlock(in_channels, out_channels, stride))
                in_channels = out_channels
        self.stages = nn.ModuleList(blocks)

        self.head = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),    # global average pooling
            nn.Flatten(),
            nn.Dropout(dropout),
            nn.Linear(in_channels, n_classes)
        )

    def forward(self, x):
        x = self.stem(x)
        for block in self.stages:
            x = block(x)
        return self.head(x)


model = AircraftNet(n_classes).to(device)
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

## Optimization

* **SGD** with Nesterov momentum 0.9 and learning rate 0.05. The linear scaling rule gives this value: 0.1 is
  tuned for batch 256, we train with batch 64.
* **Weight decay 5e-4 on weights only** — BatchNorm parameters and biases are excluded, since decaying them
  distorts the normalization statistics.
* **Cosine schedule** with 5 warmup epochs: the warmup avoids the large initial steps that a randomly initialized
  network cannot absorb, the cosine decay lets the model settle into a minimum.
* **Mixup or CutMix** on every batch, chosen at random.
* **EMA** copy of the weights, evaluated at every epoch next to the raw model.

In [ ]:
# weight decay on the weights only: BatchNorm parameters and biases are excluded
decay = [p for p in model.parameters() if p.ndim > 1]
no_decay = [p for p in model.parameters() if p.ndim <= 1]

opt = torch.optim.SGD(
    [{"params": decay, "weight_decay": 5e-4}, {"params": no_decay, "weight_decay": 0.0}],
    lr=0.05,
    momentum=0.9,
    nesterov=True
)

scaler = torch.amp.GradScaler("cuda", enabled=(device.type == "cuda"))

base_lr = 0.05
warmup_epochs = 5
label_smoothing = 0.1
ema_decay = 0.999

def ncorrect(scores, y):
    y_hat = torch.argmax(scores, -1)
    return (y_hat == y).sum()


def accuracy(scores, y):
    correct = ncorrect(scores, y)
    return correct.true_divide(y.shape[0])

In [ ]:
def learning_rate_at(epoch: int) -> float:
    """Return the learning rate of the given epoch: linear warmup, then cosine decay to zero.

    Args:
        epoch: index of the epoch.

    Returns: the learning rate to use.
    """
    if epoch < warmup_epochs:
        return base_lr * (epoch + 1) / warmup_epochs
    progress = (epoch - warmup_epochs) / max(epochs - warmup_epochs, 1)
    return base_lr * 0.5 * (1 + math.cos(math.pi * progress))


def mix_batch(imgs, labels):
    """Apply Mixup or CutMix (chosen at random) to a batch.

    Args:
        imgs: batch of images.
        labels: batch of labels.

    Returns: the mixed images, the two sets of labels, and the mixing coefficient.
    """
    use_cutmix = random.random() < 0.5
    lam = float(np.random.beta(1.0, 1.0) if use_cutmix else np.random.beta(0.2, 0.2))
    perm = torch.randperm(imgs.shape[0], device=imgs.device)

    if use_cutmix:
        h, w = imgs.shape[2:]
        box_h, box_w = int(h * math.sqrt(1 - lam)), int(w * math.sqrt(1 - lam))
        cy, cx = random.randrange(h), random.randrange(w)
        y1, y2 = max(cy - box_h // 2, 0), min(cy + box_h // 2, h)
        x1, x2 = max(cx - box_w // 2, 0), min(cx + box_w // 2, w)
        imgs[:, :, y1:y2, x1:x2] = imgs[perm][:, :, y1:y2, x1:x2]
        lam = 1 - (y2 - y1) * (x2 - x1) / (h * w)      # the area actually pasted
    else:
        imgs = lam * imgs + (1 - lam) * imgs[perm]

    return imgs, labels, labels[perm], lam


@torch.no_grad()
def update_ema(ema_model, model, decay=ema_decay):
    """Update in place the exponential moving average of the parameters.

    Args:
        ema_model: the averaged copy of the model.
        model: the model being trained.
        decay: the EMA decay; the larger, the slower the average moves.
    """
    for ema_value, value in zip(ema_model.state_dict().values(), model.state_dict().values()):
        if ema_value.dtype.is_floating_point:
            ema_value.mul_(decay).add_(value.detach(), alpha=1 - decay)
        else:
            ema_value.copy_(value)        # integer buffers, e.g. num_batches_tracked


def evaluate(model, dl):
    """Compute loss and accuracy of the model on a data loader.

    Args:
        model: the network to evaluate.
        dl: the data loader.

    Returns: the average loss and the accuracy.
    """
    model.eval()
    loss = samples = correct = 0
    with torch.no_grad():
        for data in dl:
            imgs = data[0].to(device)
            labels = data[1].to(device)
            with torch.autocast("cuda", enabled=(device.type == "cuda")):
                scores = model(imgs)
            loss += F.cross_entropy(scores, labels).item() * imgs.shape[0]
            samples += imgs.shape[0]
            correct += ncorrect(scores, labels).item()
    return loss / samples, correct / samples

In [ ]:
def train_loop(model, train_dl, epochs, opt, val_dl=None, verbose=False):
    """Train the model with the full recipe, keeping the parameters that performed best on validation.

    At every epoch both the raw model and its EMA copy are evaluated, and the better of the two is kept.

    Args:
        model: the network to train.
        train_dl: data loader of the training set.
        epochs: number of epochs.
        opt: the optimizer.
        val_dl: data loader of the validation set.
        verbose: whether to print the metrics during training.

    Returns: the best validation accuracy, the corresponding parameters, the epoch they come from,
        and the per-epoch history.
    """
    ema_model = copy.deepcopy(model)
    best_val_acc = 0
    best_params = []
    best_epoch = -1
    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": [], "ema_acc": []}

    for e in tqdm(range(epochs)):
        lr = learning_rate_at(e)
        for group in opt.param_groups:
            group["lr"] = lr

        # Train
        model.train()
        train_loss = 0
        train_samples = 0
        train_acc = 0
        for train_data in train_dl:
            imgs = train_data[0].to(device)
            labels = train_data[1].to(device)
            imgs, labels_a, labels_b, lam = mix_batch(imgs, labels)

            with torch.autocast("cuda", enabled=(device.type == "cuda")):
                scores = model(imgs)
                loss = (lam * F.cross_entropy(scores, labels_a, label_smoothing=label_smoothing)
                        + (1 - lam) * F.cross_entropy(scores, labels_b, label_smoothing=label_smoothing))

            opt.zero_grad(set_to_none=True)  # clear
            scaler.scale(loss).backward()    # fill
            scaler.step(opt)                 # use
            scaler.update()
            update_ema(ema_model, model)

            train_loss += loss.item() * imgs.shape[0]
            train_samples += imgs.shape[0]
            train_acc += ncorrect(scores, labels).item()

        train_acc /= train_samples
        train_loss /= train_samples

        # Validation: raw model and EMA copy
        val_loss, val_acc = evaluate(model, val_dl)
        _, ema_acc = evaluate(ema_model, val_dl)

        if max(val_acc, ema_acc) > best_val_acc:
            best_val_acc = max(val_acc, ema_acc)
            best_params = copy.deepcopy((ema_model if ema_acc >= val_acc else model).state_dict())
            torch.save(best_params, "best_model.pth")
            best_epoch = e

        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)
        history["ema_acc"].append(ema_acc)

        if verbose and e % 5 == 0:
            print(f"Epoch {e}: lr {lr:.4f} - train loss {train_loss:.3f} - train acc {train_acc:.3f}"
                  f" - valid loss {val_loss:.3f} - valid acc {val_acc:.3f} - ema acc {ema_acc:.3f}")

    if verbose:
        print(f"Best epoch {best_epoch}, best acc {best_val_acc:.3f}")

    return best_val_acc, best_params, best_epoch, history

## Training

In [ ]:
start_time = time.time()

best_val_acc, best_params, best_epoch, history = train_loop(
    model,
    train_dl,
    epochs,
    opt,
    val_dl,
    verbose=True
)

print(f"Training time: {(time.time() - start_time) / 60:.0f} min")

## Results

Note that **training accuracy is much lower than validation accuracy** here: it is measured on Mixup/CutMix
batches with heavy augmentation, which are far harder than clean validation images.

In [ ]:
def plot_history(history: dict, title: str) -> None:
    """Plot the loss and the accuracy recorded at every epoch.

    Args:
        history: dictionary of per-epoch lists, as returned by train_loop.
        title: title of the figure.
    """
    epochs_range = range(1, len(history["train_loss"]) + 1)
    _, ax = plt.subplots(1, 2, figsize=(13, 4.5))

    ax[0].plot(epochs_range, history["train_loss"], label="Training loss")
    ax[0].plot(epochs_range, history["val_loss"], label="Validation loss")
    ax[0].set_xlabel("Epoch"); ax[0].set_ylabel("Loss"); ax[0].grid(True); ax[0].legend()

    ax[1].plot(epochs_range, history["train_acc"], label="Training accuracy")
    ax[1].plot(epochs_range, history["val_acc"], label="Validation accuracy")
    if "ema_acc" in history:
        ax[1].plot(epochs_range, history["ema_acc"], label="Validation accuracy (EMA)")
    ax[1].set_xlabel("Epoch"); ax[1].set_ylabel("Accuracy"); ax[1].set_ylim(0, 1)
    ax[1].grid(True); ax[1].legend()

    plt.suptitle(title); plt.tight_layout(); plt.show()


plot_history(history, "Inception-style CNN")

## Test set

The test split is used **once**, on the final model. Every design decision above was taken on the validation
split only.

In [ ]:
model.load_state_dict(best_params)
model.eval()

test_loss = test_samples = test_correct = 0
confusion_matrix = torch.zeros(n_classes, n_classes, dtype=torch.int)

with torch.no_grad():
    for test_data in tqdm(test_dl):
        imgs = test_data[0].to(device)
        labels = test_data[1].to(device)
        scores = model(imgs)

        test_loss += F.cross_entropy(scores, labels).item() * imgs.shape[0]
        test_samples += imgs.shape[0]
        test_correct += ncorrect(scores, labels).item()

        for label, prediction in zip(labels, torch.argmax(scores, -1)):
            confusion_matrix[label, prediction] += 1

test_acc = test_correct / test_samples
print(f"Test - loss: {test_loss / test_samples:.4f}, accuracy: {test_acc:.3f}")

### Confusion matrix

As in Lab Session 4, we also compute the confusion matrix, and recompute the accuracy from it as a sanity check.
The most frequent confusions are informative: they are almost always *within the same family*.

In [ ]:
# accuracy recomputed from the confusion matrix, as a sanity check
print(f"Accuracy from the confusion matrix: {confusion_matrix.diag().sum() / confusion_matrix.sum():.3f}")

plt.figure(figsize=(11, 9))
sn.heatmap(pd.DataFrame(confusion_matrix), cbar=True, xticklabels=False, yticklabels=False)
plt.xlabel("Predicted"); plt.ylabel("True"); plt.title("Confusion matrix on the test set"); plt.show()

# the variants the model confuses most often
errors = confusion_matrix.clone()
errors.fill_diagonal_(0)
worst = torch.topk(errors.flatten(), 10)
print(f"{'true':28s} {'predicted':28s} {'count':>5s}")
for value, index in zip(worst.values, worst.indices):
    true_class, predicted_class = index // n_classes, index % n_classes
    print(f"{test_dset.classes[true_class]:28s} {test_dset.classes[predicted_class]:28s} {value.item():5d}")